# This notebook trains a larger Yolo model. 



In [1]:
from ultralytics import YOLO

In [20]:
model = YOLO("Yolo_models/yolo26m.pt")

### Train on public chess pieces sets


In [ ]:
model.train( # augmentation for lighting and camera place differences
    data="../../merged_dataset/data.yaml",
    epochs=100,
    hsv_v=0.5,
    degrees=30,
    translate=0.25,
    scale=0.35,
    flipud=0.3, #For pieces flips are fine
    fliplr=0.5,
    erasing=0.2,
    auto_augment="randaugment",
)


New https://pypi.org/project/ultralytics/8.4.35 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.17  Python-3.10.19 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../merged_dataset/data.yaml, degrees=30, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Yolo_models/yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000249A08AC280>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.04504

### Fine-tune to my board using hand-labels

In [ ]:
# # Balance the different classes, dataset dominated by pawns.
# # Yoinked from https://y-t-g.github.io/tutorials/yolo-class-balancing/

# import numpy as np
# import ultralytics.data as data
# import ultralytics.data.dataset as dataset

# class YOLOWeightedDataset(data.dataset.YOLODataset):
#     def __init__(self, *args, mode="train", **kwargs):
#         """
#         Initialize the WeightedDataset.

#         Args:
#             class_weights (list or numpy array): A list or array of weights corresponding to each class.
#         """

#         super(YOLOWeightedDataset, self).__init__(*args, **kwargs)

#         self.train_mode = "train" in self.prefix

#         # You can also specify weights manually instead
#         self.count_instances()
#         class_weights = np.sum(self.counts) / self.counts

#         # Aggregation function
#         self.agg_func = np.mean

#         self.class_weights = np.array(class_weights)
#         self.weights = self.calculate_weights()
#         self.probabilities = self.calculate_probabilities()
    
#     def count_instances(self):
#         """
#         Count the number of instances per class

#         Returns:
#             dict: A dict containing the counts for each class.
#         """
#         self.counts = [0 for i in range(len(self.data["names"]))]
#         for label in self.labels:
#             cls = label['cls'].reshape(-1).astype(int)
#             for id in cls:
#                 self.counts[id] += 1

#         self.counts = np.array(self.counts)
#         self.counts = np.where(self.counts == 0, 1, self.counts)

#     def calculate_weights(self):
#         """
#         Calculate the aggregated weight for each label based on class weights.

#         Returns:
#             list: A list of aggregated weights corresponding to each label.
#         """
#         weights = []
#         for label in self.labels:
#             cls = label['cls'].reshape(-1).astype(int)

#             # Give a default weight to background class
#             if cls.size == 0:
#               weights.append(1)
#               continue

#             # Take mean of weights
#             # You can change this weight aggregation function to aggregate weights differently
#             weight = self.agg_func(self.class_weights[cls])
#             weights.append(weight)
#         return weights

#     def calculate_probabilities(self):
#         """
#         Calculate and store the sampling probabilities based on the weights.

#         Returns:
#             list: A list of sampling probabilities corresponding to each label.
#         """
#         total_weight = sum(self.weights)
#         probabilities = [w / total_weight for w in self.weights]
#         return probabilities

#     def __getitem__(self, index):
#         """
#         Return transformed label information based on the sampled index.
#         """
#         # Don't use for validation
#         if not self.train_mode:
#             return self.transforms(self.get_image_and_label(index))
#         else:
#             index = np.random.choice(len(self.labels), p=self.probabilities)
#             return self.transforms(self.get_image_and_label(index))
        

# import ultralytics.data.build as build
# dataset.YOLODataset = YOLOWeightedDataset

In [2]:
model = YOLO("../../code/Training/runs/detect/train2/weights/best.pt")

model.train( #Heavy augmentation for lighting and angle differences
    data="../../project-6-at-2026-02-25-17-06-87cd7413/data.yaml",
    epochs=100,
    hsv_v=0.6,
    degrees=45,
    translate=0.25,
    scale=0.65,
    shear=5,
    perspective=0.001,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1,
    erasing=0.2,
    auto_augment="randaugment",
)


New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.17  Python-3.10.19 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../project-6-at-2026-02-25-17-06-87cd7413/data.yaml, degrees=45, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.6, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../code/Training/runs/detect/train2

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000027303FC5960>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.04504

## Sanity check through the webcam

In [4]:
# Sanity check

import cv2


model = YOLO("runs/detect/train6/weights/best.pt")
# model is assumed to already exist:
# from ultralytics import YOLO
# model = YOLO("your_model.pt")
model.to('cuda')
print(model.device)

cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO inference
    results = model(frame)

    # Draw predictions on the frame
    annotated_frame = results[0].plot()

    # Show the frame
    cv2.imshow("YOLO Webcam", annotated_frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


cuda:0

0: 480x640 (no detections), 10.1ms
Speed: 6.1ms preprocess, 10.1ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.6ms
Speed: 0.8ms preprocess, 9.6ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.9ms
Speed: 1.0ms preprocess, 9.9ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.9ms
Speed: 0.8ms preprocess, 9.9ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.5ms
Speed: 1.0ms preprocess, 9.5ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.7ms
Speed: 0.7ms preprocess, 9.7ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.7ms
Speed: 0.8ms preprocess, 9.7ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.7ms
Speed: 0.8ms preprocess, 9.7ms inferenc

In [9]:
from ultralytics import YOLO
import cv2

img = cv2.imread("../code/captured_images/image_0.jpg")
results = model(img)
cv2.imshow("Test", results[0].plot())
cv2.waitKey(0)



0: 480x640 1 BB, 1 BK, 1 BKN, 8 BPs, 1 BQ, 2 BRs, 1 WK, 1 WKN, 4 WPs, 1 WQ, 2 WRs, 12.3ms
Speed: 1.5ms preprocess, 12.3ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)


-1